# 26 · Deduplicate records, checked against rapidfuzz, jellyfish and a geodesic

## What this checks

**Deduplicate records** decides when two records from different sources are the same facility. This check covers
release 0.4.0.

It works in steps:

- **Blocking.** It sets aside pairs too far apart to consider.
- **Comparing.** It compares the rest on name and on distance. Each comparison either agrees or does not.
- **Scoring.** The score is the number of comparisons that agree. Two thresholds sort the pairs into three classes.
- **Merging.** It merges the records a match allows, at most one record from each source in any group.

No worked example uses this widget. So until this check, every claim about it rested on Fieldwork's own tests. That
is the weakest kind of evidence Fieldwork has.

## How it is checked

Four parts are redone independently. Nothing is taken from Fieldwork's code:

- **The name comparison.** Fieldwork uses its own Jaro-Winkler, a score of how alike two spellings are. We redo it
  with rapidfuzz (written in C++) and jellyfish (written in Rust), by separate authors.
- **Normalising names.** Fieldwork's rule, in JavaScript, keeps only letters and numbers. We rewrite it in plain
  Python.
- **Distances.** Fieldwork measures along a perfect sphere, of radius 6371.0088 km. We measure on the true shape of
  the Earth (a WGS84 geodesic), with pyproj.
- **Blocking, scoring, classes and groups.** We write these again from what Fieldwork declares.

Fieldwork's results are what we compare. They are never an input to our own calculation.

This notebook runs in CPython only: rapidfuzz and jellyfish are not in the browser kernel. Opened in the browser, it
says so and skips its later cells.

**The records are made up, not downloaded.** To test every rule, every branch has to fire. So the data hold each of
these:

- a pair that agrees on both comparisons;
- a pair that agrees on name alone;
- a pair that agrees on distance alone;
- a pair that agrees on neither;
- a pair too far apart to compare;
- a missing name;
- a record with no location;
- a pair from the same source;
- a group that refuses a second record from one source.

Downloaded records would hold only whichever of these they happened to contain. In all, there are 14 records, 13 with
a location.

## What should happen

1. **Name similarity** within 0.000000001 of rapidfuzz. jellyfish is carried as a third opinion.
2. **Normalised names** exactly, in both the exact and the fuzzy mode.
3. **Distances** within 1% of the geodesic. Fieldwork declares a sphere, which an earlier audit found is off by
   between −0.11% and +0.56%. An exact comparison would measure the sphere, not the code. Every pair is at least 10 m
   clear of both distance thresholds, so the classes can still be compared exactly.
4. **How many pairs** are set aside and how many compared, exactly. Pairs from the same source are neither.
5. **Each pair's** two comparisons, score and class, exactly.
6. **The merging**: how many records merge, the records that come out, and what each was merged from.

These criteria were fixed before the check was run.

In [ ]:
# Only what this notebook imports, requested one at a time, so that a package the browser kernel lacks does not stop
# the others from loading. See 00-kernel-check for what this kernel really has.
NEEDED = ["rapidfuzz", "jellyfish", "pyproj"]
try:
    import pyodide_js
except ImportError:
    pyodide_js = None
    print("Not a Pyodide kernel; using the ambient environment.")
if pyodide_js:
    for package in NEEDED:
        try:
            await pyodide_js.loadPackage(package)
            print("Loaded:", package)
        except Exception:
            print("Not in this kernel:", package)

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
try:
    from checks.check_26_deduplicate import run
except ImportError as missing:
    report = None
    print("Not runnable here:", missing)
else:
    report = run(Path.cwd() / "fixtures" / "deduplicate-facilities-001.json")
    print("criterion:", report["criterion"])
    print()
    print("agrees:", report["agrees"],
          f"(names {report['nameSimilarityAgrees']}, pipeline {report['pipelineAgrees']})")
    print(f"{report['records']} records, {report['placedRecords']} with a location")
    print(f"{report['counts']['comparedFieldwork']} pairs compared, "
          f"{report['counts']['blockedFieldwork']} never compared")
    print(f"{report['cluster']['mergedFieldwork']} records merged, "
          f"{report['cluster']['refusedCount']} matched pair(s) refused by the one-per-source rule")

## The result on record

First recorded on 25 October 2026, against release 0.1.0. Rerun on 26 November 2026, against release 0.4.0:
**Partly agrees**, with the same one difference as before.

Everything after the name scores agrees in full:

- 16 pairs compared and 26 never compared;
- every normalised name;
- each pair's two comparisons, score and class;
- 3 merges, with 2 matched pairs refused by the one-record-per-source rule;
- all 11 output records, with what each was merged from.

Distances agree within 0.497% of the geodesic, against a 1% tolerance.

One name comparison of seven differs. For `abcvwxyz` against `cabvwxyz`, Fieldwork scores 0.9375. Both reference
libraries score 0.958333, a difference of 0.020833. The section below explains why.

Running the cell above repeats the check on the same data.

## Every pair, and what each comparison said

Each comparison counts as one point or none. Scores are never blended. A name similarity of 0.87 against a threshold
of 0.88 adds nothing, not 0.87. The class follows from the score alone.

In [ ]:
if report is None:
    print("Skipped: this check is not runnable in this kernel (see above).")
else:
    print(f"{'pair':24s} {'metres':>8s} {'geodesic':>9s}  name  near  score  class")
    for row in report["pairs"]:
        pair = row["key"].replace("|", " ~ ").replace("osm:", "").replace("overture:", "")
        print(f"{pair:24s} {row['metresFieldwork']:8.1f} {row['metresReference']:9.1f} "
              f" {str(row['nameAgreesAgrees']):5s} {str(row['nearAgreesAgrees']):5s} "
              f"{str(row['scoreAgrees']):6s} {row['classFieldwork']}"
              f"{'' if row['classAgrees'] else '  <- DISAGREES'}")
    print()
    worst = max(report["pairs"], key=lambda r: r["metresRelative"])
    print(f"largest distance disagreement: {worst['metresRelative']*100:.3f}% on "
          f"{worst['key']}, against a 1% tolerance")
    print("every class agrees:", all(r["classAgrees"] for r in report["pairs"]))

## The name comparison, three ways — and one real divergence

Fieldwork, rapidfuzz and jellyfish agree exactly on every published example pair. In the literature, `martha` and
`marhta` score 0.961, and `dwayne` and `duane` score 0.840. All three give those values.

They part on one made-up case. `abcvwxyz` against `cabvwxyz` has eight matching characters, and **three** of them are
out of order. The method halves that count:

- **Fieldwork** halves it exactly, to 1.5.
- **Both libraries** halve it and drop the fraction, to 1.

So one term of the score is 6.5/8 against 7/8. The similarity is 0.9375 against 0.958333.

Two independent libraries agree with each other. That makes this a **convention, not a slip in either**. It can only
happen when the out-of-order count is odd. Every published example pair has an even count, so no published value can
catch it. No pair of names in this check's two layers reaches it.

It is recorded here, with both values, not removed.

In [ ]:
if report is None:
    print("Skipped: this check is not runnable in this kernel (see above).")
else:
    print(f"{'a':12s} {'b':12s} {'fieldwork':>10s} {'rapidfuzz':>10s} {'jellyfish':>10s}")
    for row in report["nameSimilarities"]:
        flag = "" if row["agreesWithRapidfuzz"] else "   <- diverges"
        print(f"{row['a'] or '(empty)':12s} {row['b'] or '(empty)':12s} "
              f"{row['fieldwork']:10.6f} {row['rapidfuzz']:10.6f} {row['jellyfish']:10.6f}{flag}")
    print()
    for d in report["divergences"]:
        print(f"{d['case']}: difference {d['difference']:.6f}")
        print(f"  {d['cause']}")

## The cluster rule, and the partner it chooses

A group holds at most one record from each source. Sometimes two records from one layer both match one record in the
other. Then one pairing merges. The other is refused, and stays reported as a match that did not merge.

Both sides refuse the same two pairings, because both follow the declared rule. But look at what the rule does:

- **The Annex.** Record `osm-2` is the hospital's *Annex*.
- **Its distances.** It is 2 m from Overture's record for the main hospital. It is 20 m from its own layer's record
  for the main hospital.
- **What happens.** Pairs are accepted closest first. So the Annex takes the main hospital's Overture record. The
  pairing a reader would choose is then refused.

That is a finding about the rule, not about either side's code. The widget's own documentation says such judgements
belong in the review class. But these pairs never reach review. On name and distance alone, they are matches.

In [ ]:
if report is None:
    print("Skipped: this check is not runnable in this kernel (see above).")
else:
    c = report["cluster"]
    print("merged:", c["mergedFieldwork"], "(reference", str(c["mergedReference"]) + ")")
    print("refused by the one-per-source rule:")
    for key in c["refusedEdges"]:
        print("  ", key.replace("|", " ~ "))
    print()
    print("output records agree:", c["outputRecordsAgree"],
          "· every mergedFrom agrees:", c["mergedFromAgrees"])
    print(f"{c['outputCountFieldwork']} records out of {report['records']} in")

### Reading the result

Everything except the name scores agrees. On every pair, every count, every output record and every merge, two
independent implementations give the same answer. They do so with distances from the Earth's true shape, not a
sphere.

The overall result is "partly agrees" only because of the one made-up name pair above. The lab publishes the
difference with both values, and keeps the case that found it.

## What this does not show

- What the thresholds should be. The values 0.88, 500 m and 50 m are proposals in the release, not measurements.
  Measuring them needs records whose true identity is known. These records are made up so that every branch fires.
- That closest-first grouping picks the right partner. The Annex case above shows it does not always. Both sides do
  the same, because both follow the declared rule. It is a finding about the rule.